# MeMo Step C — 3-Stage Inference Protocol on Google Colab

This notebook runs the full MEMO inference pipeline against a Memory Model
trained in Step B.

**What happens here:**
1. The trained Memory Model (Mφ) is loaded from your saved adapter.
2. A Gemini API call serves as the Executive Model (Mθ).
3. For every user question the system runs three stages:
   - **Stage 1 – Grounding**: Executive decomposes the query into atomic
     sub-questions; Memory answers each independently.
   - **Stage 2 – Entity ID**: Executive narrows to a specific entity e∗
     via iterative follow-up queries to Memory.
   - **Stage 3 – Answer Seeking + Synthesis**: Executive collects targeted
     facts about e∗, then synthesises the final answer â.

**Before running**: make sure the Colab runtime is set to GPU
(`Runtime` → `Change runtime type` → `T4 GPU`).

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Get the project into Colab

**Option A** — if you've pushed this repo to GitHub:

In [ ]:
!git clone https://github.com/sodhi4457/MeMo.git memo_project
%cd memo_project

## 3. Install Step C dependencies

Colab already has a CUDA torch installed.  The `[infer]` extra pulls
only `transformers`, `peft`, and `safetensors` — no `datasets` or
`accelerate` needed at inference time.

In [ ]:
!pip install -q -e ".[infer]"

## 4. Load the trained adapter from Google Drive

Mount Drive and point the notebook at the adapter directory saved by
Step B.  If you saved it as `memo_memory_model_ckpt`, adjust the path
below.

In [ ]:
from google.colab import drive  # type: ignore
drive.mount("/content/drive")

ADAPTER_PATH = "/content/drive/MyDrive/memo_memory_model_ckpt"
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"

import os
print(os.listdir(ADAPTER_PATH))

Expected output: a list that includes `adapter_model.safetensors`,
`adapter_config.json`, and `tokenizer.json`.

## 5. Configure the Executive Model (Gemini API)

The Executive is a **black-box API** — MEMO never touches its weights.
Paste your Google AI Studio API key below.  You can get one for free at
https://aistudio.google.com/app/apikey

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"  # <-- replace this

## 6. Load the Memory Model

The LoRA adapter is merged into the base model so there is zero overhead
at query time.  This takes ~30 seconds on a T4.

In [ ]:
from memo_infer.memory import load_memory_model

mem_model, mem_tok = load_memory_model(
    adapter_path=ADAPTER_PATH,
    base_model_name=BASE_MODEL,
)
print("Memory Model loaded and ready.")

## 7. Direct Memory probe (no Executive)

Before running the full protocol, verify that the Memory Model has
actually learned from the corpus.  These single-turn queries bypass the
Executive entirely.

In [ ]:
from memo_infer.memory import query_memory

questions = [
    "Who was Dr. Elena Marchetti and what did she study?",
    "Where did Prof. James Holbrook move after completing his doctorate?",
    "What award did Prof. Holbrook receive?",
]

for q in questions:
    a = query_memory(mem_model, mem_tok, q)
    print(f"Q: {q}")
    print(f"A: {a}")
    print()

If the answers match your corpus, the Memory Model is working.  If the
model still gives generic answers, the adapter epoch count may have been
too low — re-run Step B with more epochs.

## 8. Build the Executive callable

The Executive is created with `json_mode=False` so it can return free-
form text for synthesis stages as well as structured JSON when the
protocol prompts for it.

In [ ]:
from memo.llm_clients import get_client

executive = get_client(backend="gemini", json_mode=False)

# Quick smoke-test: verify the API key works
print(executive("Reply with exactly: OK"))

## 9. Run the full 3-stage inference protocol

Try a complex, multi-hop question that requires combining information
from multiple documents in the corpus.

In [ ]:
from memo_infer.protocol import memo_inference

USER_QUERY = (
    "What award did the scientist who collaborated with "
    "Dr. Elena Marchetti at CERN win, and when did they win it?"
)

answer = memo_inference(
    user_query=USER_QUERY,
    memory_model=mem_model,
    memory_tokenizer=mem_tok,
    executive_fn=executive,
    stage2_budget=5,   # max entity-identification turns
    stage3_budget=3,   # max answer-seeking turns
    verbose=True,      # print full stage trace
)

**Reading the trace:**
- `[STAGE 1]` shows the sub-questions the Executive decomposed and
  what the Memory Model answered for each.
- `[STAGE 2]` shows the entity-identification loop.  Look for
  `Entity identified:` to confirm convergence.
- `[STAGE 3]` shows the targeted follow-up questions.
- `[FINAL ANSWER]` is â, the synthesised answer.

## 10. Ask your own questions

In [ ]:
MY_QUESTION = "What was the most significant contribution of Dr. Elena Marchetti to physics?"

my_answer = memo_inference(
    user_query=MY_QUESTION,
    memory_model=mem_model,
    memory_tokenizer=mem_tok,
    executive_fn=executive,
    stage2_budget=5,
    stage3_budget=3,
    verbose=True,
)

## 11. (Optional) CLI equivalent

The same inference run is available from the command line.  After
installing the package the `memo-step-c` command is available:

```bash
# Full 3-stage protocol
memo-step-c query \
    --adapter /path/to/memory_model_ckpt \
    --question "What award did Holbrook receive for his CERN collaboration?"

# Direct Memory probe (no Executive)
memo-step-c probe \
    --adapter /path/to/memory_model_ckpt \
    --question "Where did Prof. Holbrook move after his doctorate?"
```

Run it inside Colab with a leading `!`:

In [ ]:
!memo-step-c probe \
    --adapter {ADAPTER_PATH} \
    --question "What field did Dr. Elena Marchetti work in?"